In [ ]:
!uv pip install langdetect

from langdetect import detect, detect_langs
from langdetect.lang_detect_exception import LangDetectException

def is_vietnamese(text):
    """
    Kiểm tra đoạn văn bản truyền vào có phải là Tiếng Việt hay không.
    """
    # Trả về False nếu văn bản trống hoặc chỉ có khoảng trắng
    if not text or not text.strip():
        return False

    try:
        # detect() trả về mã ngôn ngữ (Tiếng Việt là 'vi')
        language_code = detect(text)
        return language_code == 'vi'
    except LangDetectException:
        # Trả về False nếu không thể nhận diện được ngôn ngữ
        return False

Using Python 3.12.13 environment at: /usr
Checked 1 package in 200ms


In [ ]:
import pandas as pd

df = pd.read_csv("result/lean.csv", dtype= {
    "image_idx": "uint32",
    "file_name": "string",
    "question": "string",
    "answer": "string",
    "rewrite": "string",
    "label": "string",
})

df = df[['image_idx', 'file_name', 'rewrite', 'answer', 'label']]
df.rename(columns={'rewrite': 'question'}, inplace=True)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3560 entries, 0 to 3559
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   image_idx  3560 non-null   uint32
 1   file_name  3560 non-null   string
 2   question   3560 non-null   string
 3   answer     3560 non-null   string
 4   label      3560 non-null   string
dtypes: string(4), uint32(1)
memory usage: 125.3 KB


In [ ]:
df.at[36, 'question'] = 'Còn con vật nào khác nữa không?'
df.at[1995, 'question'] = 'Tại sao gia súc thích nghỉ ngơi dưới bóng cây?'

In [ ]:
df[~df['question'].apply(is_vietnamese)]

,image_idx,file_name,question,answer,label
2586,83324,083324.jpg,Cows are grazing in the field?,"Không, trong hình ảnh này, chúng ta thấy những...",unknown
2587,83324,083324.jpg,Cows are looking at the camera because they're...,The cows in the picture appear to be looking t...,reasoning
2594,84469,084469.jpg,Cows are grazing in the field?,"Trong hình, có bốn con bò đang ăn cỏ trên đồng...",unknown
2595,84469,084469.jpg,Cows are eating grass or not?,"Trong hình, có hai con bò đang ăn cỏ trong đồng.",reasoning
2596,84493,084493.jpg,Cows are grazing in the field?,"Trong hình này, có một con bò đang nằm trên đồ...",object
...,...,...,...,...,...
3538,166535,166535.jpg,Cows are bigger or smaller than sheep?,Cows are generally larger than sheep.,comparison
3539,166535,166535.jpg,Cows are grazing in the field.,"Nhìn vào hình ảnh này, chúng ta thấy một đàn b...",object
3541,166844,166844.jpg,Cows are bigger or smaller than the poles?,"Trong hình ảnh này, bò lớn hơn so với các cột....",comparison
3556,167948,167948.jpg,Cows are eating grass?,"Đúng rồi, những con bò trong hình đang ăn cỏ t...",unknown


In [ ]:
df_rewrite = pd.read_csv("rewrited.csv", dtype= {
    "id_question": "uint32",
    "origin_question": "string",
    "LLM_labled": "string",
    "rewrite": "string"
})

df_merge = df.merge(
    df_rewrite.set_index('id_question'),
    left_index=True,      # Giữ nguyên Index 0-3559 của df làm khóa
    right_index=True, # Khớp với giá trị trong cột id_question
    how='left'            # Giữ đủ 3560 dòng của df
)
# df_merge[df_merge['rewrite'].notna()]
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3560 entries, 0 to 3559
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   image_idx        3560 non-null   uint32 
 1   file_name        3560 non-null   string 
 2   question         3560 non-null   string 
 3   answer           3560 non-null   string 
 4   label            3560 non-null   string 
 5   origin_question  78 non-null     string 
 6   LLM_labeled      78 non-null     object 
 7   rewrite          78 non-null     string 
 8   rewrite_len      78 non-null     float64
dtypes: float64(1), object(1), string(6), uint32(1)
memory usage: 236.5+ KB


In [ ]:
# 1. Tạo mặt nạ (mask) lọc các dòng có rewrite (không phải NaN)
mask = df_merge['rewrite'].notna()

# 2. Gán giá trị từ rewrite sang question chỉ tại những vị trí đó
df_merge.loc[mask, 'question'] = df_merge.loc[mask, 'rewrite']

# 3. Kiểm tra kết quả (in ra các dòng đã được cập nhật)
df = df_merge[['image_idx', 'file_name', 'question', 'answer', 'label']].copy()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3560 entries, 0 to 3559
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   image_idx  3560 non-null   uint32
 1   file_name  3560 non-null   string
 2   question   3560 non-null   string
 3   answer     3560 non-null   string
 4   label      3560 non-null   string
dtypes: string(4), uint32(1)
memory usage: 125.3 KB


In [ ]:
df.at[3462, 'question'] = 'Bò gặm cỏ khô là mùa nào?'
df[df["question"].str.split().str.len() > 10]
df.to_csv('semi-lean.csv', index=False)

In [ ]:
import pandas as pd

df = pd.read_csv("semi-lean.csv", dtype= {
    "image_idx": "uint32",
    "file_name": "string",
    "question": "string",
    "answer": "string",
    "label": "string"
})

In [ ]:
import numpy as np

# 1. Lấy danh sách các image_idx duy nhất và xáo trộn
unique_ids = df['image_idx'].unique()
np.random.shuffle(unique_ids)

# 2. Tính toán điểm chia (tỉ lệ 8/1/1)
n = len(unique_ids)
train_end = int(n * 0.8)
val_end = int(n * 0.9)

# 3. Chia danh sách ID
train_ids = unique_ids[:train_end]
val_ids = unique_ids[train_end:val_end]
test_ids = unique_ids[val_end:]

# 4. Tạo các DataFrame con từ danh sách ID tương ứng
train_df = df[df['image_idx'].isin(train_ids)].reset_index(drop=True)
val_df = df[df['image_idx'].isin(val_ids)].reset_index(drop=True)
test_df = df[df['image_idx'].isin(test_ids)].reset_index(drop=True)

# Kiểm tra kết quả
print(f"Train: {len(train_df)} dòng, {len(train_ids)} ảnh")
print(f"Val:   {len(val_df)} dòng, {len(val_ids)} ảnh")
print(f"Test:  {len(test_df)} dòng, {len(test_ids)} ảnh")

Train: 2834 dòng, 773 ảnh
Val:   346 dòng, 97 ảnh
Test:  380 dòng, 97 ảnh


In [ ]:
train_df['answer'] = [[x] for x in train_df['answer']]
with open('train.json', 'w', encoding='utf-8') as f:
    f.write(train_df.reset_index().to_json(orient='records', force_ascii=False, indent=2))

val_df['answer'] = [[x] for x in val_df['answer']]
with open('origin_val.json', 'w', encoding='utf-8') as f:
    f.write(val_df.reset_index().to_json(orient='records', force_ascii=False, indent=2))

test_df['answer'] = [[x] for x in test_df['answer']]
with open('origin_test.json', 'w', encoding='utf-8') as f:
    f.write(test_df.reset_index().to_json(orient='records', force_ascii=False, indent=2))


In [ ]:
!uv pip install bitsandbytes accelerate qwen-vl-utils

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

class QwenVLBatchProcessor:
    def __init__(self, model_id="Qwen/Qwen2.5-VL-7B-Instruct"):
        """Khởi tạo mô hình tối ưu cho GPU T4"""
        print(f"Đang tải mô hình {model_id}...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )

        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto"
        )
        self.processor = AutoProcessor.from_pretrained(model_id)

        # SỬA ĐỔI 1: Sử dụng từ điển (dictionary) để quản lý mã lệnh cho nhiều nhiệm vụ
        self.system_prompts = {}
        print("Khởi tạo hoàn tất.")

    def set_system_prompt(self, task_name, system_prompt):
        """
        SỬA ĐỔI 2: Thiết lập mã lệnh hệ thống (system prompt) theo từng nhiệm vụ.
        Ví dụ task_name có thể là 'answer' hoặc 'generate_question'.
        """
        self.system_prompts[task_name] = system_prompt
        print(f"Đã cập nhật System Prompt cho nhiệm vụ: {task_name}")

    def _prepare_image_source(self, image_source):
        """Hàm nội bộ xử lý đường dẫn ảnh"""
        if str(image_source).startswith("/"):
            return f"file://{image_source}"
        return image_source

    def generate_response(self, image_source, question, max_new_tokens=128, extract_keyword="Câu trả lời:"):
        """
        Nhiệm vụ: Trả lời câu hỏi dựa trên ảnh và tự động bóc tách kết quả.
        Tham số extract_keyword mặc định là 'Câu trả lời:' để tự động cắt bỏ tiền tố rườm rà.
        """
        image_source = self._prepare_image_source(image_source)

        sys_prompt = self.system_prompts.get("answer", "Bạn là chuyên gia phân tích hình ảnh AI.")

        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": [
                {"type": "image", "image": image_source},
                {"type": "text", "text": question}
            ]}
        ]

        # Nhận kết quả thô từ mô hình
        raw_output = self._inference(messages, max_new_tokens)

        # Gọi thẳng hàm parse_response bên trong class để làm sạch dữ liệu
        return self._parse_response(raw_output, extract_keyword)

    def _parse_response(self, raw_response, extract_keyword=None):
        """Hàm nội bộ giúp làm sạch và bóc tách phản hồi"""
        clean_text = raw_response.strip()

        if extract_keyword and extract_keyword in clean_text:
            return clean_text.split(extract_keyword)[-1].strip()

        return clean_text

    def generate_question_by_label(self, image_source, label, existing_questions=None, max_new_tokens=50):
        """
        SỬA ĐỔI 3: Sinh câu hỏi không trùng lặp và không bắt buộc liên quan đến ảnh.
        Tham số existing_questions nhận vào một danh sách (list) các câu hỏi đã tạo.
        """
        image_source = self._prepare_image_source(image_source)

        # Lấy mã lệnh tương ứng với nhiệm vụ tạo dữ liệu
        sys_prompt = self.system_prompts.get("generate_question", "Bạn là chuyên gia tạo dữ liệu.")

        # Xây dựng yêu cầu cốt lõi (User Request)
        user_request = f"Hãy đặt một câu hỏi tiếng Việt thuộc loại nhãn '{label}'. Câu hỏi này KHÔNG bắt buộc phải liên quan trực tiếp đến các chi tiết có trong hình ảnh. Đảm bảo câu hỏi tự nhiên và dưới 10 từ đơn."

        # Xử lý logic chống trùng lặp nếu có dữ liệu cũ
        if existing_questions and len(existing_questions) > 0:
            # Nối các câu hỏi cũ thành một danh sách văn bản
            danh_sach_cu = "\n- ".join(existing_questions)
            user_request += f"\n\nTuyệt đối KHÔNG tạo câu hỏi trùng lặp ý nghĩa hoặc cấu trúc với các câu sau:\n- {danh_sach_cu}"

        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": [
                {"type": "image", "image": image_source},
                {"type": "text", "text": user_request}
            ]}
        ]

        raw_output = self._inference(messages, max_new_tokens)
        return raw_output.replace("Câu hỏi:", "").strip()

    def _inference(self, messages, max_new_tokens, do_sample=False, temperature=None, repetition_penalty=None):
        """Hàm suy luận nội bộ: Ép buộc xóa sạch dấu vết tham số mặc định để chặn cảnh báo."""
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(self.model.device)

        # 1. Khởi tạo gói tham số sinh văn bản (Generation Parameters)
        gen_kwargs = {
            "max_new_tokens": max_new_tokens,
            "do_sample": do_sample,
        }

        # 2. Phân luồng kiểm soát thông số ngẫu nhiên
        if do_sample:
            if temperature is not None:
                gen_kwargs["temperature"] = temperature
            gen_kwargs["top_p"] = 0.85
            gen_kwargs["top_k"] = 40
        else:
            # Chủ động đè giá trị rỗng lên các tham số mặc định trong balo của mô hình
            gen_kwargs["temperature"] = None
            gen_kwargs["top_p"] = None
            gen_kwargs["top_k"] = None

        # 3. Hình phạt lặp từ (Repetition Penalty)
        if repetition_penalty is not None:
            gen_kwargs["repetition_penalty"] = repetition_penalty

        with torch.no_grad():
            # Truyền tham số bằng kỹ thuật Unpacking
            generated_ids = self.model.generate(**inputs, **gen_kwargs)

        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]

        return self.processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()

    def generate_diverse_answer(self, image_source, question, existing_answers=None, fails=None, max_new_tokens=128, extract_keyword="Câu trả lời:"):
        """
        Nhiệm vụ: Trả lời câu hỏi dựa trên ảnh, không trùng lặp câu cũ và có cơ chế tự né tránh câu trả lời hỏng.
        """
        image_source = self._prepare_image_source(image_source)
        sys_prompt = self.system_prompts.get("diverse_answer", "Bạn là chuyên gia phân tích hình ảnh AI.")
        user_request = f"Câu hỏi: {question}"

        # Xử lý logic chống trùng lặp với các câu trả lời hợp lệ đã có trong cơ sở dữ liệu
        if existing_answers and len(existing_answers) > 0:
            danh_sach_cu = "\n- ".join(existing_answers)
            user_request += f"\n\nYêu cầu bắt buộc: Dựa vào hình ảnh, hãy trả lời câu hỏi trên. Tuyệt đối KHÔNG được lặp lại ý tưởng hoặc thông tin của các câu sau:\n- {danh_sach_cu}\n\nHãy tìm một chi tiết khác hoặc góc nhìn khác để trả lời."

        kich_hoat_lay_mau = False
        nhiet_do = None
        hinh_phat = None

        # Xử lý logic tự nắn chỉnh khi tạo ra câu trả lời bị lỗi định dạng hoặc không đạt chuẩn
        if fails and len(fails) > 0:
            last_fail = fails[-1]

            # Lời nhắc cảnh báo được thiết kế linh hoạt, tránh giới hạn cứng nhắc về vật thể
            user_request += f"\n\n[CẢNH BÁO LỖI]: Lần thử trước bạn đã đưa ra câu trả lời là '{last_fail}' nhưng hệ thống không chấp nhận. Hãy quan sát hình ảnh lại một lần nữa, linh hoạt thay đổi hướng diễn đạt và đưa ra một câu trả lời hoàn toàn mới. Tuyệt đối không đi vào lối mòn của câu trả lời sai vừa rồi."

            kich_hoat_lay_mau = True
            hinh_phat = 1.15

            # Tăng mức độ sáng tạo dựa trên độ bướng bỉnh của mô hình
            muc_do_gia_tang = len(fails) * 0.15
            nhiet_do = min(0.95, 0.4 + muc_do_gia_tang)

        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": [
                {"type": "image", "image": image_source},
                {"type": "text", "text": user_request}
            ]}
        ]

        raw_output = self._inference(
            messages,
            max_new_tokens=max_new_tokens,
            do_sample=kich_hoat_lay_mau,
            temperature=nhiet_do,
            repetition_penalty=hinh_phat
        )

        return self._parse_response(raw_output, extract_keyword)

qwen_processor = QwenVLBatchProcessor()

Using Python 3.12.13 environment at: /usr
Checked 3 packages in 725ms
Đang tải mô hình Qwen/Qwen2.5-VL-7B-Instruct...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Khởi tạo hoàn tất.


In [ ]:
import json
from pathlib import Path
import re
from tqdm.auto import tqdm
import copy

# Khai báo mã lệnh hệ thống cho nhiệm vụ trả lời đa dạng
DIVERSE_ANSWER_SYSTEM_PROMPT = """
Bạn là chuyên gia phân tích hình ảnh AI cao cấp, chuyên gia tạo dữ liệu đa góc nhìn.
Nhiệm vụ: Phân tích hình ảnh và trả lời câu hỏi được đưa ra. Câu hỏi có thể sử dụng bất kỳ ngôn ngữ nào (Tiếng Anh, Tiếng Việt, v.v.), nhưng bạn PHẢI luôn luôn trả lời bằng Tiếng Việt một cách tự nhiên và chuyên sâu. Hãy tìm ra các chi tiết mới mẻ, độc đáo hoặc diễn đạt theo một hướng tiếp cận khác biệt hoàn toàn so với các quan sát thông thường.

Lưu ý:
- Phải trả lời đúng trọng tâm câu hỏi, không bịa đặt thông tin ngoài bức ảnh.
- Nếu không còn chi tiết nào mới để khai thác, hãy mở rộng ngữ cảnh, giải thích sâu về ý nghĩa/tác động của chi tiết đó, hoặc thay đổi hoàn toàn cấu trúc câu để tạo sự mới lạ.
- Ngôn ngữ phản hồi: Tiếng Việt thuần túy, tự nhiên.
- Luôn tuân thủ định dạng: Câu trả lời: <nội dung phản hồi>
""".strip()

def add_more_answer(input_file, output_file):
    print(f"--- BẮT ĐẦU BỔ SUNG CÂU TRẢ LỜI CHO {input_file}")

    # Thiết lập mã lệnh hệ thống cho nhiệm vụ trả lời đa dạng
    qwen_processor.set_system_prompt("diverse_answer", DIVERSE_ANSWER_SYSTEM_PROMPT)

    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        a_per_q = 10
        max_attempt = 20

        # Duyệt qua từng phần tử và thêm giá trị vào list answer
        with tqdm(total=len(data), desc="Processing", position=0) as pbar:
            for item in data:
                img = str((Path('images') / Path(item['file_name'])).resolve())

                with tqdm(total=a_per_q, desc=f"Processing {item['index']}", position=1, leave=False) as p:
                    p.update(len(item['answer']))
                    attempt = 0

                    # Khởi tạo danh sách hứng lỗi cho phiên làm việc hiện tại
                    fails = []

                    while len(item['answer']) < a_per_q and attempt < max_attempt:
                        p.set_postfix(retry=f'{attempt}/{max_attempt}')

                        # Truyền danh sách lỗi vào hàm để kích hoạt cơ chế tự sửa sai
                        new_a = qwen_processor.generate_diverse_answer(
                            image_source=img,
                            question=item['question'],
                            existing_answers=item['answer'],
                            fails=fails
                        )

                        if new_a and is_vietnamese(new_a) and new_a not in item['answer']:
                            item['answer'].append(new_a)
                            p.update(1)
                            # Dọn sạch thùng rác khi đã tìm được câu trả lời hợp lệ
                            # fails.clear()
                        else:
                            tqdm.write(f"Invalid [{new_a}] trong {input_file}, index: {item['index']}")
                            # Tống câu trả lời hỏng vào danh sách để làm bài học cho lượt sau
                            fails.append(new_a)
                            attempt += 1
                    else:
                        tqdm.write(f"- Chỉ tạo được {len(item['answer'])}/{a_per_q} câu trả lời cho {input_file}, index: {item['index']}")
                        pbar.update(1)
                        # break

        # Lưu dữ liệu đã chỉnh sửa xuống file mới
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"--- KẾT THÚC BỔ SUNG CÂU TRẢ LỜI CHO {input_file}")

add_more_answer("origin_val.json", "val.json")
add_more_answer("origin_test.json", "test.json")

--- BẮT ĐẦU BỔ SUNG CÂU TRẢ LỜI CHO origin_val.json
Đã cập nhật System Prompt cho nhiệm vụ: diverse_answer


Processing:   0%|          | 0/346 [00:00<?, ?it/s]

Processing 0:   0%|          | 0/10 [00:00<?, ?it/s]

Câu trả lời không hợp lệ: Trong hình, có tổng cộng 6 con bò. Một con bò lớn đang đứng giữa, hai con bò nhỏ bên trái và bên phải, và hai con bò khác ở phía xa hơn. Chi tiết thú vị khác là con bò lớn ở giữa có vẻ đang tập trung nhìn vào phía trước, trong khi hai con bò nhỏ bên cạnh nó dường như đang quan sát người chụp ảnh. trong origin_val.json, index: 0
Câu trả lời không hợp lệ: Có bảy con bò trong hình. Một con bò lớn đang đứng giữa, hai con bò nhỏ bên trái và bên phải, và hai con bò khác ở phía xa hơn. Chi tiết thú vị khác là con bò lớn ở giữa có vẻ đang tập trung nhìn vào phía trước, tạo nên cảm giác như nó đang quan sát mọi thứ xung quanh mình. trong origin_val.json, index: 0
Chỉ tạo được 10 câu trả lời cho origin_val.json, index: 0


Processing 1:   0%|          | 0/10 [00:00<?, ?it/s]

...

Processing 8:   0%|          | 0/10 [00:00<?, ?it/s]

Câu trả lời không hợp lệ: Trong hình ảnh này, ngoài hai chú bê đang đứng gần nhau, chúng ta cũng có thể thấy một con bò khác ở phía xa hơn, đang đi qua cánh đồng. Chi tiết này cho thấy môi trường sống tự nhiên của những con vật này, nơi chúng có thể di chuyển tự do. Ngoài ra, dưới chân của con bê màu nâu, có một con bê màu đen đang đứng yên, tạo nên một khung cảnh thú vị về sự tương tác giữa các loài vật trong tự nhiên. trong origin_val.json, index: 8
Chỉ tạo được 10 câu trả lời cho origin_val.json, index: 8


Processing 9:   0%|          | 0/10 [00:00<?, ?it/s]

Câu trả lời không hợp lệ: Những chú bê đang ở trong một khu vực có nhiều cây cối và đất sét, gần như là một con đường nhỏ dẫn qua cánh đồng cỏ xanh mướt. Chi tiết mới mẻ hơn là những chú bê này đang ở trong một môi trường tự nhiên, nơi chúng có thể thoải mái di chuyển và kiếm ăn, đồng thời cũng tạo nên một khung cảnh yên bình và thân thiện với thiên nhiên. trong origin_val.json, index: 9
Câu trả lời không hợp lệ: Những chú bê đang ở trong một khu vực có nhiều cây cối và đất sét, gần như là một con đường nhỏ dẫn qua cánh đồng cỏ xanh mướt. Chi tiết mới mẻ hơn là những chú bê này đang ở trong một môi trường tự nhiên, nơi chúng có thể thoải mái di chuyển và kiếm ăn, đồng thời cũng tạo nên một khung cảnh yên bình và thân thiện với thiên nhiên. trong origin_val.json, index: 9


KeyboardInterrupt: 

In [ ]:
import json

# 1. Đọc dữ liệu từ file gốc
with open('origin_test.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. Duyệt qua từng phần tử và thêm giá trị vào list answer
for item in data:
    # Giả sử bạn muốn thêm một câu trả lời mới vào list answer hiện có
    new_ans = "Đây là câu trả lời bổ sung"
    item['answer'].append(new_ans)

# 3. Lưu dữ liệu đã chỉnh sửa xuống file mới
with open('test.json', 'w', encoding='utf-8') as f:
    # indent=2 giúp file dễ đọc, ensure_ascii=False để hiển thị tiếng Việt chuẩn
    json.dump(data, f, ensure_ascii=False, indent=2)

print("Đã cập nhật và lưu file thành công!")


Đã cập nhật và lưu file thành công!


In [ ]:
df_rewrite = pd.read_csv("rewrited.csv", dtype= {
    "image_idx": "uint32",
    "file_name": "string",
    "question": "string",
    "answer": "string",
    "rewrite": "string",
    "label": "string",
})

In [ ]:
df_rewrite[df_rewrite['id_question'] == 3462]

,id_question,origin_question,LLM_labeled,rewrite,rewrite_len
74,3462,"Cows are grazing in a dry field, what season m...",reasoning,Might it be autumn?,4


In [ ]:
df_rewrite.at[74, 'rewrite'] = 'Bò gặm cỏ trên đồng khô, có thể là mùa gì?'

In [ ]:
df_rewrite.to_csv('rewrited.csv', index=False)